# W11-D4 概念实验：KnowledgeSnapshot 与可复现性

核心概念来自 Markdown：当前 RAG 工程能力较强，但 KnowledgeCollection 与 KnowledgeSnapshot 没有分离，运行时直接消费可变集合。下面用内容寻址 digest 模拟“集合变更必须产生新部署闭包”。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
import hashlib
import json

def snapshot_digest(documents, config):
    payload = {"documents": sorted(documents), "config": config}
    canonical = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return "ks-" + hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]

collection = ["招商政策：免租期上限90天", "合同审批：招商主管复核"]
config = {"chunk_size": 500, "embedding": "demo-embedding-v1", "top_k": 5}
digest_v1 = snapshot_digest(collection, config)
digest_same = snapshot_digest(list(reversed(collection)), config)
digest_v2 = snapshot_digest(collection + ["政策生效日：2026-08-01"], config)
print("相同内容（顺序不同）digest:", digest_v1, digest_same, digest_v1 == digest_same)
print("内容变化后 digest:", digest_v2, digest_v1 != digest_v2)

In [ ]:
revisions = ["dr-2026-08-01"]
pinned_snapshot = digest_v1
print({"deployment_revision": revisions[0], "knowledge_snapshot_digest": pinned_snapshot})
print("运行时读取规则：只接受 pinned digest，不直接读取可变 collection。")
try:
    collection.append("临时修改：免租期上限60天")
    assert snapshot_digest(collection, config) == pinned_snapshot
except AssertionError:
    print("检测到集合已变更：必须创建新的 KnowledgeSnapshot，并生成新的 DeploymentRevision。")

In [ ]:
changes = ["初始集合", "新增政策文档", "调整 chunk_size"]
digests = [digest_v1, digest_v2, snapshot_digest(collection, {**config, "chunk_size": 800})]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(range(len(changes)), [1, 2, 3], "o-", color="#4c78a8")
ax.set_xticks(range(len(changes)), changes)
ax.set_yticks(range(1, 4), [d[-8:] for d in digests])
ax.set_ylabel("快照 digest（后8位）")
ax.set_title("知识变更与不可变快照链")
for i, d in enumerate(digests):
    ax.annotate(d[-8:], (i, i + 1), textcoords="offset points", xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()
plt.close(fig)
print("结论：RAG 检索优化解决“找得准”，Snapshot 解决“用的是哪一版”；两者是不同治理维度。")